# 01_relations.py — Entity relations: ownership, Inverse, partial loading

Run the setup first, then each step in order. This notebook executes the same declarations, calls and assertions as the [script](01_relations.py). Outputs were captured by direct Python execution in a fresh process. No Jupyter kernel was used.


In [ ]:
!pip install -q aoa-action-machine

In [2]:
from __future__ import annotations

from typing import Annotated

from pydantic import Field

from aoa.action_machine.domain import (
    AssociationMany,
    AssociationOne,
    BaseCoreEntity,
    BaseDetailEntity,
    CompositeMany,
    Inverse,
    RelationNotLoadedError,
)
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.entities import entities
from aoa.action_machine.intents.entity import entity
from aoa.action_machine.runtime.action_product_machine import ActionProductMachine
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.resources.base_storage import BaseStorage


class ShopDomain(BaseDomain):
    name = "shop"
    description = "Shop domain"


@entity(description="Customer account", domain=ShopDomain)
class CustomerCoreEntity(BaseCoreEntity):
    id: str = Field(description="Customer id")
    name: str = Field(description="Display name")

    orders: Annotated[
        AssociationMany[OrderCoreEntity],
        Inverse(field_name="customer"),
    ] = Field(description="Orders placed by this customer")


@entity(description="Customer order", domain=ShopDomain)
class OrderCoreEntity(BaseCoreEntity):
    id: str = Field(description="Order id")
    total: float = Field(ge=0, description="Order total")

    # association — order and customer exist independently
    customer: Annotated[
        AssociationOne[CustomerCoreEntity],
        Inverse(CustomerCoreEntity, "orders"),
    ] = Field(description="Customer who placed the order")

    # composition — line items cannot exist without this order
    lines: Annotated[
        CompositeMany[OrderLineDetailEntity],
        Inverse(field_name="order"),
    ] = Field(description="Line items of the order")


@entity(description="Single line of an order", domain=ShopDomain)
class OrderLineDetailEntity(BaseDetailEntity):
    id: str = Field(description="Line id")
    sku: str = Field(description="Product code")

    order: Annotated[
        AssociationOne[OrderCoreEntity],
        Inverse(OrderCoreEntity, "lines"),
    ] = Field(description="Parent order")


@meta(description="Stores the shop entities")
@entities(
    CustomerCoreEntity,
    OrderCoreEntity,
    OrderLineDetailEntity,
)
class ShopStorage(BaseStorage):
    """Hold the shop entities."""


CustomerCoreEntity.model_rebuild()
OrderCoreEntity.model_rebuild()
OrderLineDetailEntity.model_rebuild()

## Actionproductmachine


In [3]:
ActionProductMachine()

## Inspect the result


In [4]:
print("1) Machine built — relation model validated at startup (Inverse mirroring OK)")

1) Machine built — relation model validated at startup (Inverse mirroring OK)


## Run the next step


In [5]:
order = OrderCoreEntity(
    id="ord-1",
    total=1500.0,
    customer=AssociationOne[CustomerCoreEntity](id="cust-1"),
    lines=CompositeMany[OrderLineDetailEntity](ids=("line-1", "line-2")),
)

## Inspect the result


In [6]:
print("\n2) Id-only relations:")


2) Id-only relations:


## Inspect the result


In [7]:
print(f"   order.customer.id  = {order.customer.id}   (is_loaded={order.customer.is_loaded})")

   order.customer.id  = cust-1   (is_loaded=False)


## Inspect the result


In [8]:
print(f"   len(order.lines)   = {len(order.lines)}   (is_loaded={order.lines.is_loaded})")

   len(order.lines)   = 2   (is_loaded=False)


## Check the expected refusal and assert its diagnostic


In [9]:
try:
    _ = order.customer.name  # target row not hydrated
except RelationNotLoadedError as exc:
    print(f"   order.customer.name -> RelationNotLoadedError: {exc}")

   order.customer.name -> RelationNotLoadedError: Related object in AssociationOne is not loaded (id='cust-1'). Cannot access 'name' — only the identifier is present. Load the related entity through your persistence / manager layer.


## Check the expected refusal and assert its diagnostic


In [10]:
try:
    _ = list(order.lines)  # many: iteration requires loaded entities
except RelationNotLoadedError as error:
    print("   iterating order.lines -> RelationNotLoadedError:", error)
else:
    raise AssertionError("Unloaded collection was iterable")

   iterating order.lines -> RelationNotLoadedError: Related object in CompositeMany is not loaded (id=('line-1', 'line-2')). Cannot access '__iter__' — only the identifier is present. Load the related entity through your persistence / manager layer.


## Run the next step


In [11]:
customer = CustomerCoreEntity(
    id="cust-1",
    name="Alice",
    orders=AssociationMany[OrderCoreEntity](ids=("ord-1",)),
)

## Run the next step


In [12]:
order2 = OrderCoreEntity(
    id="ord-2",
    total=99.0,
    customer=AssociationOne[CustomerCoreEntity](id="cust-1", entity=customer),
    lines=CompositeMany[OrderLineDetailEntity](entities=(), entities_loaded=True),
)

## Verify the result


In [13]:
assert order2.customer.name == "Alice"

## Verify the result


In [14]:
assert order2.lines.is_loaded

## Inspect the result


In [15]:
print("\n3) Hydrated relation:")


3) Hydrated relation:


## Inspect the result


In [16]:
print(f"   order2.customer.name = {order2.customer.name}   (proxied through the loaded entity)")

   order2.customer.name = Alice   (proxied through the loaded entity)
